In [21]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader,Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)
import os

In [9]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


Mounted at /content/drive


In [10]:
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'Screenshot_2023-07-23-19-35-54-287_com.android.chrome.jpg', 'IMG-20230723-WA0040.jpg', 'IMG-20240322-WA0007.jpg', 'IMG-20240322-WA0008.jpg', 'IMG-20240322-WA0009.jpg', 'DocScanner 22-Mar-2024 8-58 pm-1(16958185477625).jpg', 'DocScanner 22-Mar-2024 8-58 pm-2(16958321498507).jpg', 'IMG-20240322-WA0010.jpg', 'IMG-20240322-WA0011.jpg', 'SHEETAL AADHAR.pdf', 'Priority A Chapters PYQs_Eduniti.gdoc', 'Priority B Chapters PYQs_Eduniti.gdoc', 'Priority C Chapters PYQs_Eduniti.gdoc', ' BackZero gp I, II and III gp practicals.pdf', 'CHEM PRACTICAL IV grp.pdf', 'Untitled video.gvid', 'Classroom', 'Introduction to Electric Circuits, 9th Edition - Richard C. Dorf, James A. Svoboda.pdf', 'ADDITIONAL INSTRUCTIONS FOR 10+2 B.Tech-ENTRY .gdoc', 'SIH.gdoc', 'practice_NUMPY.ipynb', 'pandas_learning.ipynb', 'WhatsApp Image 2026-03-06 at 12.00.10 AM.jpeg', 'Machine_Learning', 'Fetching API, WEB SCRAPING.ipynb', 'day_20_univariate analysis.ipynb', 'Untitled0.ipynb', 'WhatsApp Image 2026-

In [11]:
ROOT_DIR = "/content/drive/MyDrive/SER_MEL_SPECTROGRAMS"

df = pd.read_csv(
    os.path.join(ROOT_DIR,"labels.csv")
)

df.head()

,File,Label,Emotion
0,0.npy,5,neutral
1,1.npy,5,neutral
2,2.npy,5,neutral
3,3.npy,5,neutral
4,4.npy,1,calm


In [19]:
train_df,test_df=train_test_split(
    df,
    test_size=0.2,
    stratify=df["Label"],
    random_state=42
)
print(train_df.shape)
print(test_df.shape)

(1152, 3)
(288, 3)


In [22]:
class SpectrogramDataset(Dataset):

    def __init__(self, dataframe, root_dir):

        self.dataframe = dataframe
        self.root_dir = root_dir

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        mel = np.load(
            os.path.join(
                self.root_dir,
                row["File"]
            )
        )
        mel = torch.tensor(
            mel,
            dtype=torch.float32)
        mel = mel.unsqueeze(0)
        label = torch.tensor(
            row["Label"],
            dtype=torch.long
        )

        return mel, label

In [23]:
train_dataset=SpectrogramDataset(train_df,ROOT_DIR)
test_dataset=SpectrogramDataset(test_df,ROOT_DIR)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {device}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using Device: cuda
Tesla T4


In [61]:
class EmotionCNN(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.features=nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(128 * 16 * 16, 512),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(512,256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256,num_classes)
    )
  def forward(self,x):
    x=self.features(x)
    x=self.classifier(x)

    return x

In [62]:
num_classes=df['Label'].nunique()
model=EmotionCNN(num_classes).to(device)

print(model)

EmotionCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=512, bias=True)
    (2): ReLU()
    (3): Drop

In [63]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [64]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [65]:
epochs=50
train_losses=[]
for epoch in range(epochs):
  model.train()
  running_loss=0.0
  for images,labels in train_loader:
    images=images.to(device,non_blocking=True)
    labels=labels.to(device,non_blocking=True)

    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
  epoch_loss=running_loss/len(train_loader)
  train_losses.append(epoch_loss)
  print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/50, Loss: 3.5495
Epoch 2/50, Loss: 1.9523
Epoch 3/50, Loss: 1.7898
Epoch 4/50, Loss: 1.6875
Epoch 5/50, Loss: 1.6615
Epoch 6/50, Loss: 1.5715
Epoch 7/50, Loss: 1.4739
Epoch 8/50, Loss: 1.3945
Epoch 9/50, Loss: 1.2890
Epoch 10/50, Loss: 1.3385
Epoch 11/50, Loss: 1.2516
Epoch 12/50, Loss: 1.1549
Epoch 13/50, Loss: 1.0406
Epoch 14/50, Loss: 0.9843
Epoch 15/50, Loss: 0.9560
Epoch 16/50, Loss: 0.8642
Epoch 17/50, Loss: 0.8412
Epoch 18/50, Loss: 0.8338
Epoch 19/50, Loss: 0.7961
Epoch 20/50, Loss: 0.7154
Epoch 21/50, Loss: 0.6980
Epoch 22/50, Loss: 0.7013
Epoch 23/50, Loss: 0.6552
Epoch 24/50, Loss: 0.6412
Epoch 25/50, Loss: 0.5239
Epoch 26/50, Loss: 0.4897
Epoch 27/50, Loss: 0.5483
Epoch 28/50, Loss: 0.5526
Epoch 29/50, Loss: 0.5369
Epoch 30/50, Loss: 0.5437
Epoch 31/50, Loss: 0.5198
Epoch 32/50, Loss: 0.4967
Epoch 33/50, Loss: 0.4575
Epoch 34/50, Loss: 0.5660
Epoch 35/50, Loss: 0.4873
Epoch 36/50, Loss: 0.4912
Epoch 37/50, Loss: 0.4654
Epoch 38/50, Loss: 0.4381
Epoch 39/50, Loss: 0.

In [66]:
model.eval()
predictions=[]
actual=[]
with torch.no_grad():
  for images,labels in test_loader:
    images=images.to(device,non_blocking=True)
    labels=labels.to(device,non_blocking=True)
    outputs=model(images)
    preds=torch.argmax(outputs,dim=1)
    predictions.extend(preds.cpu().numpy())
    actual.extend(labels.cpu().numpy())

In [67]:
model.eval()

train_preds = []
train_labels = []

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

print("Training Accuracy:",
      accuracy_score(train_labels, train_preds))

Training Accuracy: 0.9887152777777778


In [68]:
print(f"Accuracy : {accuracy_score(actual, predictions):.4f}")
print(classification_report(actual, predictions))
print(confusion_matrix(actual, predictions))

Accuracy : 0.6979
              precision    recall  f1-score   support

           0       0.83      0.79      0.81        38
           1       0.69      0.87      0.77        38
           2       0.70      0.84      0.76        38
           3       0.62      0.79      0.70        39
           4       0.74      0.44      0.55        39
           5       0.61      0.74      0.67        19
           6       0.61      0.29      0.39        38
           7       0.75      0.85      0.80        39

    accuracy                           0.70       288
   macro avg       0.69      0.70      0.68       288
weighted avg       0.70      0.70      0.68       288

[[30  0  4  1  0  0  1  2]
 [ 0 33  0  0  0  2  3  0]
 [ 2  1 32  0  0  0  1  2]
 [ 1  3  0 31  3  0  0  1]
 [ 3  0  2  8 17  3  1  5]
 [ 0  4  0  1  0 14  0  0]
 [ 0  7  6  6  3  4 11  1]
 [ 0  0  2  3  0  0  1 33]]


In [38]:
images, labels = next(iter(train_loader))

print(images.shape)
print(images.min())
print(images.max())
print(images.mean())
print(images.std())

torch.Size([64, 1, 128, 128])
tensor(-1.3315)
tensor(4.5746)
tensor(1.5367e-08)
tensor(1.0000)


In [39]:
print(np.unique(predictions))

[0 1 2 3 4 5 6]


In [40]:
print(df["Label"].value_counts().sort_index())

Label
0    192
1    192
2    192
3    192
4    192
5     96
6    192
7    192
Name: count, dtype: int64
